In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# How to Evaluate Propositional Formulas

In this section, we want to show how to evaluate formulas from <em>propositional logic</em>.  First of all we need to discuss how to represent logical formulas.  We represent formulas as values of the *discriminated union* `Formula` that is defined in the notebook `Propositional-Logic-Parser.ipynb`:
```
type Formula =
    | Var    of string             // propositional variable
    | Verum                        // ⊤
    | Falsum                       // ⊥
    | Not    of Formula            // ¬
    | And    of Formula * Formula  // ∧
    | Or     of Formula * Formula  // ∨
    | Imp    of Formula * Formula  // →
    | Iff    of Formula * Formula  // ↔
    | Xor    of Formula * Formula  // ⊕
```
When we write formulas as strings, we use the following unicode symbols for the propositional operators:
<table border="1px">
    <tr>
        <th>Name</th>
        <th>Symbol</th>
        <th>Unicode</th>
        <th>Unicode Name</th>
    </tr>
    <tr>
        <td><tt>and</tt></td>
        <td>∧</td>
        <td>U+2227</td>
        <td><tt>logical and</tt></td>
    </tr>
    <tr>
        <td><tt>or</tt></td>
        <td>∨</td>
        <td>U+2228</td>
        <td><tt>logical or</tt></td>
    </tr>
    <tr>
        <td><tt>not</tt></td>
        <td>¬</td>
        <td>U+00ac</td>
        <td><tt>not sign</tt></td>
    </tr>
    <tr>
        <td><tt>implies</tt></td>
        <td>→</td>
        <td>U+2192</td>
        <td><tt>rightwards arrow</tt></td>
    </tr>
    <tr>
        <td><tt>if and only if</tt></td>
        <td>↔</td>
        <td>U+2194</td>
        <td><tt>left right arrow</tt></td>
    </tr>
    <tr>
        <td><tt>falsum</tt></td>
        <td>⊥</td>
        <td>U+22a5</td>
        <td><tt>down tack</tt></td>
    </tr>
    <tr>
        <td><tt>verum</tt></td>
        <td>⊤</td>
        <td>U+22a4</td>
        <td><tt>up tack</tt></td>
    </tr>
</table>
For example, the formula 
$$ (p \rightarrow q) \wedge (\neg p \rightarrow q) \rightarrow q $$
will be represented as the following value of type `Formula`:
```
Imp (And (Imp (Var "p", Var "q"), Imp (Not (Var "p"), Var "q")), Var "q")
```
In F#, a unicode character of the form `U+abcd` can be written as `ꯍ` inside a string.

In [2]:
"p∧q"

"p∧q"

For any Unicode code point (including those outside the *Basic Multilingual Plane*, up to `U+10FFFF`), we can use the `\UXXXXXXXX` escape sequence, where 
`XXXXXXXX` represents the eight hexadecimal digits of the code point (padding with leading zeros if necessary).

In [3]:
"\U00002227"

"∧"

In [4]:
"\U0001F600"

"😀"

It is also possible to just copy and paste these symbols.  That is what I usually do.
Another possibility is to use $\LaTeX$ to create these symbols.  We have
* `$\top$` is rendered as $\top$,
* `$\bot$` is rendered as $\bot$,
* `$\neg$` is rendered as $\neg$,
* `$\wedge$` is rendered as $\wedge$,
* `$\vee$` is rendered as $\vee$,
* `$\rightarrow$` is rendered as $\rightarrow$,
* `$\leftrightarrow$` is rendered as $\leftrightarrow$.

As it is quite tedious to type formulas as values of type `Formula`, I have written a parser for propositional formulas, which we will import.
The parser will correctly parse an well-formed string that represents a formula from propositional logic.  However, it will not detect 
certain syntax errors:
  * The parser assumes that the operator $\wedge$ binds stronger than the operator $\vee$.
  
    Therefore, it parses the string $p \vee q \wedge r$ as if it was the formula $p \vee (q \wedge r)$.
    
    According to the conventions of our lecture, this string should be rejected as syntactically invalid.
    
  * The parser assumes that the operator $\leftrightarrow$ is right *associative*.
  
    Therefore, it parses the string $p \leftrightarrow q \leftrightarrow r$ as if it was the formula $p \leftrightarrow (q \leftrightarrow r)$.
    
    According to the conventions of our lecture, this string should be rejected as syntactically invalid.

The function `importNotebook` is defined in the file `setup.fsx`.  It executes all definitions of the given notebook.  The definitions are available once the cell has finished.

In [5]:
importNotebook "Propositional-Logic-Parser.ipynb"

imported 

Propositional-Logic-Parser.ipynb

The parser provides the function `parse` that takes a string and returns the corresponding formula.  Let us try to parse a complex formula.

In [6]:
let s = "(p → q) ∧ (¬p → q) → q"
let f = parse s
f

Imp (And (Imp (Var "p", Var "q"), Imp (Not (Var "p"), Var "q")), Var "q")

Next, we need to decide how to represent a *propositional valuation*.  Mathematically, a *propositional valuation* $\mathcal{I}$ is a function of the form
$$ \mathcal{I}:\mathcal{P} \rightarrow \mathbb{B}  $$
that maps propositional variables to the set of Boolean values 
$\mathbb{B} = \{\texttt{true}, \texttt{false}\}$.  We can represent a propositional  valuation $\mathcal{I}$ as the set of those variables that are evaluated as <tt>true</tt> by $\mathcal{I}$:  
$$ \texttt{repr}(\mathcal{I}) := \bigl\{ v \in \mathcal{P} \mid \mathcal{I}(v) = \texttt{true} \bigr\}$$
Hence, in F# a propositional valuation has the type `Set<string>`.
For example, the following set would be a propositional valuation for the formula $\texttt{f}$:

In [7]:
let I1 = set [ "p"; "q" ]

There are 4 possible propositional valuation for the formula $\texttt{f}$.  The other three propositional valuations are as follows:

In [8]:
let I2 = set [ "p" ]
let I3 = set [ "q" ]
let I4 : Set<string> = Set.empty

In F#, the Boolean values are ordered: `false` is less than `true`.  Therefore, the implication $a \rightarrow b$ is true if and only if $a \leq b$.

In [9]:
false < true

true

The function `evaluate` defined below takes two arguments:
   - `f` is a *propositional formula*.
   - `I` is a set of *propositional variables* representing a *propositional valuation*.
   
The function returns the value of the propositional formula `f` given the propositional valuation `I`.

**Implementation:** The function is defined by *pattern matching*: For every constructor of the type `Formula` there is a case that specifies how a formula of this form is evaluated.  The F# compiler checks that the patterns cover all constructors.  If we forgot a case, the compiler would issue a warning.
- A variable `p` is true if and only if it is an element of the set `I`.
- When evaluating a formula of the form $g \rightarrow h$, the function uses the fact that the implication is true iff the value of $g$ is less than or equal to the value of $h$.
- A formula of the form $g \leftrightarrow h$ is true iff $g$ and $h$ have the same value.
- A formula of the form $g \oplus h$ is true iff $g$ and $h$ have different values.

In [10]:
let rec evaluate (f: Formula) (I: Set<string>) : bool =
    match f with
    | Var p      -> I.Contains p
    | Verum      -> true
    | Falsum     -> false
    | Not g      -> not (evaluate g I)
    | And (g, h) -> evaluate g I && evaluate h I
    | Or  (g, h) -> evaluate g I || evaluate h I
    | Imp (g, h) -> evaluate g I <= evaluate h I
    | Iff (g, h) -> evaluate g I =  evaluate h I
    | Xor (g, h) -> evaluate g I <> evaluate h I

Let's test this function for the formula 
$$ f := (p \rightarrow q) \wedge (\neg p \rightarrow q) \rightarrow q. $$

In [11]:
f

Imp (And (Imp (Var "p", Var "q"), Imp (Not (Var "p"), Var "q")), Var "q")

In [12]:
for I in [I1; I2; I3; I4] do
    printfn "evaluate f %s = %b" (show I) (evaluate f I)

evaluate f 

set ["p"; "q"]

 = 

true

evaluate f 

...

true

evaluate f 

set []

 = 

true

As the formula <tt>f</tt> is true for **every** possible propositional valuation `I`, it is a *tautology*.